# Save LoRA Adapters from Trained Model

This notebook loads a trained LoRA model and saves ONLY the adapter weights (before any merging).

**Purpose:** Fix empty adapter files by saving adapters from training checkpoints.

**Input:** Training checkpoint from `./output/biblical_llama31_8b_instruct_unsloth_4bit/train`

**Output:** Valid LoRA adapter files for vLLM deployment

In [2]:
# Configuration
MODEL_NAME_BASE = "biblical_llama31_8b_instruct_unsloth_4bit"
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
TRAINING_CHECKPOINT_DIR = f"{OUTPUT_BASE_DIR}/train"  # Where training saved checkpoints
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters_fixed"  # New output directory
VLLM_LORAS_DIR = "/sdb-disk/vllm_loras/biblical_llama31_8b_instruct_unsloth_4bit"

print(f"✓ Will load checkpoint from: {TRAINING_CHECKPOINT_DIR}")
print(f"✓ Will save adapters to: {LORA_OUTPUT_DIR}")
print(f"✓ Will copy to vLLM at: {VLLM_LORAS_DIR}")

✓ Will load checkpoint from: ./output/biblical_llama31_8b_instruct_unsloth_4bit/train
✓ Will save adapters to: ./output/biblical_llama31_8b_instruct_unsloth_4bit/lora_adapters_fixed
✓ Will copy to vLLM at: /sdb-disk/vllm_loras/biblical_llama31_8b_instruct_unsloth_4bit


In [3]:
# Check what checkpoints exist
import os
from pathlib import Path

checkpoint_dirs = [d for d in Path(TRAINING_CHECKPOINT_DIR).iterdir() if d.is_dir() and d.name.startswith('checkpoint')]
checkpoint_dirs.sort(key=lambda x: int(x.name.split('-')[1]) if '-' in x.name else 0)

print(f"Found {len(checkpoint_dirs)} checkpoints:")
for cp in checkpoint_dirs:
    print(f"  - {cp.name}")

if checkpoint_dirs:
    LATEST_CHECKPOINT = str(checkpoint_dirs[-1])
    print(f"\n✓ Will use latest: {LATEST_CHECKPOINT}")
else:
    print("\n⚠️  No checkpoints found. Make sure training completed.")

Found 6 checkpoints:
  - checkpoint-200
  - checkpoint-400
  - checkpoint-600
  - checkpoint-800
  - checkpoint-1000
  - checkpoint-1151

✓ Will use latest: output/biblical_llama31_8b_instruct_unsloth_4bit/train/checkpoint-1151


In [4]:
# Load the PEFT model (adapter only, not merged)
from peft import PeftModel, PeftConfig
from transformers import AutoTokenizer

print("📥 Loading PEFT adapter configuration...")
config = PeftConfig.from_pretrained(LATEST_CHECKPOINT)

print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(LATEST_CHECKPOINT)

print(f"\n✓ Adapter config loaded")
print(f"  Base model: {config.base_model_name_or_path}")
print(f"  LoRA r: {config.r}")
print(f"  LoRA alpha: {config.lora_alpha}")
print(f"  Target modules: {config.target_modules}")

/home/spark/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu130 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
/home/spark/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


📥 Loading PEFT adapter configuration...
📥 Loading tokenizer...

✓ Adapter config loaded
  Base model: unsloth/meta-llama-3.1-8b-instruct-bnb-4bit
  LoRA r: 16
  LoRA alpha: 32
  Target modules: {'v_proj', 'q_proj'}


In [5]:
# Save ONLY the LoRA adapters (no base model, no merging)
import shutil

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"💾 Copying LoRA adapter files to {LORA_OUTPUT_DIR}...")

# Copy adapter files from checkpoint
adapter_files = [
    'adapter_config.json',
    'adapter_model.safetensors',
]

for file in adapter_files:
    src = Path(LATEST_CHECKPOINT) / file
    dst = Path(LORA_OUTPUT_DIR) / file
    if src.exists():
        shutil.copy2(src, dst)
        size = dst.stat().st_size
        print(f"  ✓ Copied {file} ({size:,} bytes)")
    else:
        print(f"  ⚠️  {file} not found in checkpoint")

# Also copy tokenizer files
print("\n💾 Saving tokenizer...")
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"\n✅ LoRA adapters saved to {LORA_OUTPUT_DIR}")

💾 Copying LoRA adapter files to ./output/biblical_llama31_8b_instruct_unsloth_4bit/lora_adapters_fixed...
  ✓ Copied adapter_config.json (1,131 bytes)
  ✓ Copied adapter_model.safetensors (27,280,152 bytes)

💾 Saving tokenizer...

✅ LoRA adapters saved to ./output/biblical_llama31_8b_instruct_unsloth_4bit/lora_adapters_fixed


In [ ]:
# Verify adapter file size
adapter_file = Path(LORA_OUTPUT_DIR) / "adapter_model.safetensors"
if adapter_file.exists():
    size = adapter_file.stat().st_size
    print(f"📊 adapter_model.safetensors size: {size:,} bytes ({size/1024/1024:.2f} MB)")
    
    if size < 1000:
        print("\n⚠️  WARNING: File is suspiciously small (< 1KB). Adapters may be empty!")
    elif size < 100000:
        print("\n⚠️  File seems small. Expected a few MB for r=16 adapters.")
    else:
        print("\n✓ File size looks reasonable for LoRA adapters.")
else:
    print("❌ adapter_model.safetensors not found!")